# Hydris - Driver Engines (Phase 1)
**Goal:** compute the *why* behind every Aqueduct headline score - drought, groundwater and flood
driving factors from raw satellite data, with per-driver attribution %, per the three toolkit docs.

**Sub-phases + gates:**
- **1a Drought** `D_H = 0.35*M + 0.30*A + 0.35*H` (meteorological / agricultural / hydrological)
- **1b Groundwater** `GH = 0.35*EP + 0.30*AD + 0.20*RW (+0.15*QD excluded)` - weights renormalized
- **1c Flood** 5 factors: rainfall, SCS-CN runoff, catchment, soil saturation, urbanization

**No-hallucination rules (enforced in code):**
1. Every dataset ID is *probed live* in GEE before use (cell 2). Unavailable -> driver marked `no-data`, weights renormalized. IDs are never assumed.
2. Every driver carries `source` + `confidence` (`computed` / `proxy` / `regional` / `no-data`).
3. Attribution is labeled *"estimated contribution to the modelled score"* - never physical causation.
4. GRACE is always labeled `regional` (~300 km signal - basin-scale only, never site-scale).

In [ ]:
!pip install -q earthengine-api numpy scipy pandas
import ee, json, time
import numpy as np
import scipy.stats as stats

EE_PROJECT = "your-gee-project-id"   # <-- EDIT
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

AQ = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/baseline_annual")

SITES = [
    {"id":"S1","name":"Chennai plant (IN)","lat":13.0827,"lng":80.2707},
    {"id":"S2","name":"Tiruppur textile (IN)","lat":11.1085,"lng":77.3411},
    {"id":"S3","name":"Fresno plant (US-CA)","lat":36.7378,"lng":-119.7871},
    {"id":"S4","name":"Hamburg plant (DE)","lat":53.5511,"lng":9.9937},
    {"id":"S5","name":"Riyadh plant (SA)","lat":24.7136,"lng":46.6753},
]

CHIRPS_YEARS = list(range(1995, 2026))   # CHIRPS solid from 1981; 30-yr window
COMMON_YEARS = list(range(2003, 2026))   # MODIS/GLDAS/GRACE-safe window
print("setup ok")

## 2. Dataset probe - verify every ID *live* before any engine uses it
Candidates come from the three toolkit docs + GEE catalog. Whatever fails to load is recorded as
`None` and the dependent driver degrades to `no-data` with renormalized weights.
For HYSOGs250m the docx says: check the community catalog first, else download from ORNL DAAC
(DOI 10.3334/ORNLDAAC/1566) and upload as `projects/YOUR_PROJECT/assets/HYSOGs250m`.

In [ ]:
def probe_ic(cands, band=None):
    """First ImageCollection candidate that loads (and has `band` if given), else None."""
    for cid in cands:
        try:
            ic = ee.ImageCollection(cid)
            bands = ic.first().bandNames().getInfo()
            if band and band not in bands: continue
            return cid
        except Exception: continue
    return None

def probe_img(cands, band=None):
    for cid in cands:
        try:
            img = ee.Image(cid)
            bands = img.bandNames().getInfo()
            if band and band not in bands: continue
            return cid
        except Exception: continue
    return None

DS = {}
DS["chirps"]    = probe_ic(["UCSB-CHG/CHIRPS/DAILY"], "precipitation")
DS["gldas"]     = probe_ic(["NASA/GLDAS/V021/NOAH/G025/T3H"], "SoilMoi0_10cm_inst")
DS["ndvi"]      = probe_ic(["MODIS/061/MOD13Q1","MODIS/006/MOD13Q1"], "NDVI")
DS["pet"]       = probe_ic(["MODIS/061/MOD16A2GF","MODIS/061/MOD16A2","MODIS/006/MOD16A2"], "PET")
DS["grace"]     = probe_ic(["NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI","NASA/GRACE/MASS_GRIDS_V04/MASCON",
                             "NASA/GRACE/MASS_GRIDS/MASCON_CRI","NASA/GRACE/MASS_GRIDS/LAND"], "lwe_thickness")
DS["smap"]      = probe_ic(["NASA/SMAP/SPL4SMGP/008","NASA/SMAP/SPL4SMGP/007"], "sm_surface")
DS["worldcover"]= probe_ic(["ESA/WorldCover/v200","ESA/WorldCover/v100"], "Map")
DS["ghsl"]      = probe_ic(["JRC/GHSL/P2023A/GHS_BUILT_S"], "built_surface")
DS["copdem"]    = probe_ic(["COPERNICUS/DEM/GLO30"], "DEM")
DS["flowacc"]   = probe_img(["WWF/HydroSHEDS/15ACC"], "b1")
DS["condem"]    = probe_img(["WWF/HydroSHEDS/15CONDEM"], "b1")
DS["sand"]      = probe_img(["projects/soilgrids-isric/sand_mean",
                              "OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02"])
# HYSOGs250m: community-catalog candidates first, then your own upload (edit the last entry)
DS["hysogs"]    = probe_img([
    "projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m",
    "projects/sat-io/open-datasets/HYSOGS250m",
    f"projects/{EE_PROJECT}/assets/HYSOGs250m",
])

for k, v in DS.items():
    print(("OK      " if v else "MISSING ") + f"{k:<11} -> {v}")
print("\nAny MISSING entry -> that driver reports confidence='no-data' and weights renormalize.")
print("hysogs MISSING -> flood runoff ships as 'pending asset' per Gate 1c rule (download from ORNL DAAC, upload, re-run).")

## 3. Shared plumbing
- `basin_geom` - Aqueduct basin polygon for a point (same basin the headline score uses)
- `aq_scores` - Aqueduct scores for proxy/comparison use
- `yearly` - N-year basin-mean series in **one** `getInfo()` round-trip (server-side map over years)
- `zscore`, `gamma_spi`, severity mappings, and the composite+attribution builder

In [ ]:
def clean(v):
    return None if v in (-9999, -9999.0, "-9999") else v

def basin_geom(lat, lng):
    hit = AQ.filterBounds(ee.Geometry.Point([lng, lat]))
    if hit.size().getInfo() == 0: return None, None
    f = hit.first()
    return f.geometry(), f.get("pfaf_id").getInfo()

def aq_scores(lat, lng, keys=("bws_score","bwd_score","gtd_score","drr_score","rfr_score","cfr_score")):
    hit = AQ.filterBounds(ee.Geometry.Point([lng, lat]))
    if hit.size().getInfo() == 0: return {}
    d = hit.first().toDictionary(list(keys)).getInfo()
    return {k: clean(d.get(k)) for k in keys}

def yearly(col_id, band, geom, months, temporal_reducer, scale, years, mult=1.0):
    """Per-year basin means of an N-month window ending Dec 31. ONE getInfo round-trip.
    Returns list aligned with `years`; missing years are None.

    Empty-window safe: years before a dataset starts (e.g. MOD16 PET < 2000) or inside
    mission gaps (GRACE 2017-18) produce an EMPTY filterDate -> 0-band composite, and any
    image math on that errors server-side ('Image.multiply: ... Got 0 and 1'). So each
    year is guarded by ee.Algorithms.If on the window's image count (If evaluates lazily -
    the dead branch is never computed), and the unit scaling (`mult`) is applied
    client-side to the returned numbers, never to the image."""
    col = ee.ImageCollection(col_id).select(band)
    def per_year(y):
        y = ee.Number(y)
        end = ee.Date.fromYMD(y, 12, 31)
        start = end.advance(-months, "month")
        sub = col.filterDate(start, end)
        d = sub.reduce(temporal_reducer).reduceRegion(
                ee.Reducer.mean(), geom, scale, maxPixels=1e9, bestEffort=True)
        v = ee.Algorithms.If(
                sub.size().gt(0),
                ee.Algorithms.If(d.size().gt(0), d.values().get(0), None),
                None)
        return ee.Feature(None, {"v": v})
    fc = ee.FeatureCollection(ee.List(years).map(per_year))
    feats = fc.getInfo()["features"]
    return [None if (v := f["properties"].get("v")) is None else v * mult for f in feats]

def zscore(series):
    """z of the latest non-None value vs the whole series. None if too thin."""
    vals = [v for v in series if v is not None]
    if len(vals) < 8: return None
    a = np.array(vals, float)
    if a.std() == 0: return None
    return float((a[-1] - a.mean()) / a.std())

def gamma_spi(series):
    """Proper SPI: gamma fit on positive totals + zero-mass correction -> normal quantile.
    Falls back to None if the fit is not viable (caller then uses zscore)."""
    vals = [v for v in series if v is not None]
    if len(vals) < 12: return None
    x = np.array(vals, float)
    pos = x[x > 0]
    if len(pos) < 10: return None
    try:
        q0 = float((x <= 0).mean())
        a, loc, b = stats.gamma.fit(pos, floc=0)
        cdf = q0 + (1 - q0) * stats.gamma.cdf(max(x[-1], 1e-9), a, loc=0, scale=b)
        cdf = float(np.clip(cdf, 1e-4, 1 - 1e-4))
        return float(stats.norm.ppf(cdf))
    except Exception:
        return None

def sev_dry(z):   # deficit variables: more negative z = more severe
    return None if z is None else float(np.clip(0.5 - z/4.0, 0, 1))

def sev_wet(z):   # excess variables: more positive z = more severe
    return None if z is None else float(np.clip(0.5 + z/4.0, 0, 1))

def composite(drivers):
    """drivers: {name: {severity, weight, source, confidence, ...}}.
    Renormalizes weights over available severities; returns hazard + attribution %."""
    avail = {k: d for k, d in drivers.items() if d.get("severity") is not None}
    if not avail:
        return {"hazard_0_1": None, "hazard_0_5": None, "attribution_pct": {},
                "drivers": drivers, "excluded": list(drivers)}
    wsum = sum(d["weight"] for d in avail.values())
    hazard = sum(d["severity"] * d["weight"] for d in avail.values()) / wsum
    contrib = {k: d["severity"] * d["weight"] for k, d in avail.items()}
    total = sum(contrib.values()) or 1.0
    attribution = {k: round(100 * c / total, 1) for k, c in contrib.items()}
    return {
        "hazard_0_1": round(hazard, 3),
        "hazard_0_5": round(hazard * 5, 2),
        "attribution_pct": attribution,
        "attribution_note": "estimated contribution to the modelled score (not physical causation)",
        "drivers": drivers,
        "excluded": [k for k in drivers if k not in avail],
    }

print("plumbing ready (empty-window-safe yearly)")

## 1a - Drought engine
`D_H = 0.35*M + 0.30*A + 0.35*H`, each pillar split evenly across its drivers:
- **M** meteorological: SPI (gamma-fit, CHIRPS 3-mo; z-score fallback) + SPEI-lite (z of P - PET, MOD16)
- **A** agricultural: GLDAS 0-10 cm soil-moisture anomaly + MODIS NDVI anomaly
- **H** hydrological: GLDAS runoff anomaly + GRACE storage anomaly (`regional` flag)

In [ ]:
def _drv(sev, weight, source, confidence, **extra):
    d = {"severity": None if sev is None else round(sev, 3),
         "weight": weight, "source": source, "confidence": confidence}
    d.update(extra); return d

def drought_drivers(lat, lng):
    t0 = time.time()
    geom, pfaf = basin_geom(lat, lng)
    if geom is None: return None
    out = {}

    # M1: SPI - CHIRPS 3-month totals, gamma-fit with z fallback
    if DS["chirps"]:
        rain = yearly(DS["chirps"], "precipitation", geom, 3, ee.Reducer.sum(), 5000, CHIRPS_YEARS)
        spi = gamma_spi(rain); method = "gamma-SPI"
        if spi is None: spi = zscore(rain); method = "zscore-fallback"
        out["rainfall_SPI"] = _drv(sev_dry(spi), 0.175, DS["chirps"], "computed",
                                    index=None if spi is None else round(spi,2), method=method)
    else:
        out["rainfall_SPI"] = _drv(None, 0.175, "CHIRPS unavailable", "no-data")

    # M2: SPEI-lite - z of (P - PET) 3-month
    if DS["chirps"] and DS["pet"]:
        pet = yearly(DS["pet"], "PET", geom, 3, ee.Reducer.sum(), 1000, CHIRPS_YEARS, mult=0.1)  # 0.1 -> mm
        pmpet = [None if (r is None or p is None) else r - p for r, p in zip(rain, pet)]
        z = zscore(pmpet)
        out["water_balance_SPEI"] = _drv(sev_dry(z), 0.175, f'{DS["chirps"]} + {DS["pet"]}', "computed",
                                          index=None if z is None else round(z,2), method="z of P-PET (SPEI-lite)")
    else:
        out["water_balance_SPEI"] = _drv(None, 0.175, "MOD16 PET unavailable", "no-data")

    # A1: soil moisture
    if DS["gldas"]:
        sm = yearly(DS["gldas"], "SoilMoi0_10cm_inst", geom, 3, ee.Reducer.mean(), 25000, COMMON_YEARS)
        z = zscore(sm)
        out["soil_moisture"] = _drv(sev_dry(z), 0.15, DS["gldas"], "computed",
                                     index=None if z is None else round(z,2))
    else:
        out["soil_moisture"] = _drv(None, 0.15, "GLDAS unavailable", "no-data")

    # A2: vegetation (NDVI)
    if DS["ndvi"]:
        nd = yearly(DS["ndvi"], "NDVI", geom, 3, ee.Reducer.mean(), 1000, COMMON_YEARS)
        z = zscore(nd)
        out["vegetation_NDVI"] = _drv(sev_dry(z), 0.15, DS["ndvi"], "computed",
                                       index=None if z is None else round(z,2))
    else:
        out["vegetation_NDVI"] = _drv(None, 0.15, "MODIS NDVI unavailable", "no-data")

    # H1: runoff
    if DS["gldas"]:
        ro = yearly(DS["gldas"], "Qs_acc", geom, 3, ee.Reducer.mean(), 25000, COMMON_YEARS)
        z = zscore(ro)
        out["runoff"] = _drv(sev_dry(z), 0.175, DS["gldas"], "computed",
                              index=None if z is None else round(z,2))
    else:
        out["runoff"] = _drv(None, 0.175, "GLDAS unavailable", "no-data")

    # H2: GRACE storage - REGIONAL signal only
    if DS["grace"]:
        gr = yearly(DS["grace"], "lwe_thickness", geom, 12, ee.Reducer.mean(), 100000, COMMON_YEARS)
        z = zscore(gr)
        out["storage_GRACE"] = _drv(sev_dry(z), 0.175, DS["grace"], "regional",
                                     index=None if z is None else round(z,2),
                                     caveat="~300 km resolution - basin/regional signal, never site-scale")
    else:
        out["storage_GRACE"] = _drv(None, 0.175, "GRACE unavailable", "no-data")

    res = composite(out)
    res.update({"risk": "drought", "pfaf_id": pfaf, "runtime_s": round(time.time()-t0, 1)})
    return res

print("drought engine defined")

In [ ]:
# ✅ GATE 1a - Chennai / Fresno / Hamburg. PASS requires:
#   attribution sums to ~100; gamma-SPI method reported; directional agreement with Aqueduct drr;
#   runtime logged per site.
for name, lat, lng in [("Chennai",13.0827,80.2707), ("Fresno",36.7378,-119.7871), ("Hamburg",53.5511,9.9937)]:
    d = drought_drivers(lat, lng)
    aq = aq_scores(lat, lng)
    ssum = sum(d["attribution_pct"].values())
    print(f"\n=== {name} ===  runtime {d['runtime_s']}s")
    print(f"  modelled drought hazard: {d['hazard_0_5']} /5   | Aqueduct drr_score: {aq.get('drr_score')}")
    print(f"  attribution (sum={ssum:.1f}%):")
    for k, pct in sorted(d["attribution_pct"].items(), key=lambda x: -x[1]):
        drv = d["drivers"][k]
        print(f"    {k:<20} {pct:>5}%  sev={drv['severity']}  [{drv['confidence']}] {drv.get('method','')}")
    if d["excluded"]: print("  excluded (no-data):", d["excluded"])
    assert abs(ssum - 100) < 1.5, "attribution must sum to ~100%"
print("\nGATE 1a: assertions passed - check directional agreement + runtimes above by eye")

## 1b - Groundwater engine
`GH = 0.35*EP + 0.30*AD + 0.20*RW + 0.15*QD` ->
- **EP** extraction pressure: **proxy** from Aqueduct `gtd`/`bwd` (no global well/abstraction dataset
  exists - production integration: CGWB (India) / USGS NWIS (US); labeled as such)
- **AD** aquifer decline: GRACE `linearFit` trend over the basin (`regional`)
- **RW** recharge worry: 1 - recharge-potential index (CHIRPS rain x sand fraction x slope x landcover)
- **QD** quality: **excluded** - no global groundwater-chemistry raster; weights renormalize over 0.85

In [ ]:
# landcover infiltration coefficients (0-1, higher = more infiltration) - heuristic lookup
# per the groundwater toolkit's recharge method, applied to ESA WorldCover classes
LC_INFIL = {10:.80, 20:.70, 30:.65, 40:.60, 50:.20, 60:.50, 70:.50, 80:1.0, 90:.90, 95:.90, 100:.60}

def grace_trend_cm_yr(geom):
    col = (ee.ImageCollection(DS["grace"]).select("lwe_thickness")
           .filterDate("2003-01-01", "2025-12-31"))
    def addt(img):
        t = img.date().difference(ee.Date("2003-01-01"), "year")
        return ee.Image.constant(t).float().rename("t").addBands(img)
    fit = col.map(addt).select(["t", "lwe_thickness"]).reduce(ee.Reducer.linearFit())
    d = fit.select("scale").reduceRegion(ee.Reducer.mean(), geom, 100000, maxPixels=1e9, bestEffort=True).getInfo()
    return d.get("scale")

def recharge_index(geom):
    """0-1 recharge potential: 0.40*rain + 0.25*sand + 0.20*landcover-infiltration + 0.15*flatness."""
    parts, srcs = {}, []
    if DS["chirps"]:
        p = (ee.ImageCollection(DS["chirps"]).select("precipitation")
             .filterDate("2005-01-01","2025-01-01").sum().divide(20)
             .reduceRegion(ee.Reducer.mean(), geom, 5000, maxPixels=1e9, bestEffort=True).getInfo())
        p_ann = list(p.values())[0] if p else None
        if p_ann is not None:
            parts["rain"] = (float(np.clip(p_ann/1500.0, 0, 1)), 0.40); srcs.append(DS["chirps"])
    if DS["sand"]:
        img = ee.Image(DS["sand"])
        band = img.bandNames().getInfo()[0]
        sv = img.select(band).reduceRegion(ee.Reducer.mean(), geom, 1000, maxPixels=1e9, bestEffort=True).getInfo().get(band)
        if sv is not None:
            frac = sv/1000.0 if sv > 100 else sv/100.0   # soilgrids g/kg vs openlandmap %
            parts["sand"] = (float(np.clip(frac, 0, 1)), 0.25); srcs.append(DS["sand"])
    if DS["worldcover"]:
        lc = (ee.ImageCollection(DS["worldcover"]).first()
              .reduceRegion(ee.Reducer.mode(), geom, 1000, maxPixels=1e9, bestEffort=True).getInfo().get("Map"))
        if lc is not None:
            parts["landcover"] = (LC_INFIL.get(int(lc), 0.5), 0.20); srcs.append(DS["worldcover"])
    dem_src = DS["copdem"] or DS["condem"]
    if dem_src:
        dem = ee.ImageCollection(DS["copdem"]).select("DEM").mosaic() if DS["copdem"] else ee.Image(DS["condem"])
        sl = (ee.Terrain.slope(dem)
              .reduceRegion(ee.Reducer.mean(), geom, 1000, maxPixels=1e9, bestEffort=True).getInfo())
        slope = list(sl.values())[0] if sl else None
        if slope is not None:
            parts["flatness"] = (float(np.clip(1 - slope/30.0, 0, 1)), 0.15); srcs.append(dem_src)
    if not parts: return None, srcs
    wsum = sum(w for _, w in parts.values())
    return sum(v*w for v, w in parts.values())/wsum, srcs

def groundwater_drivers(lat, lng):
    t0 = time.time()
    geom, pfaf = basin_geom(lat, lng)
    if geom is None: return None
    aq = aq_scores(lat, lng)
    out = {}

    # EP - PROXY from Aqueduct (documented gap: needs site abstraction / CGWB / USGS NWIS)
    ep_parts = [v for v in (aq.get("gtd_score"), aq.get("bwd_score")) if v is not None]
    ep = (sum(ep_parts)/len(ep_parts))/5.0 if ep_parts else None
    out["extraction_pressure"] = _drv(ep, 0.35, "WRI Aqueduct gtd+bwd", "proxy",
        caveat="proxy - site abstraction data required for full EP (CGWB / USGS NWIS in production)")

    # AD - GRACE trend (regional)
    if DS["grace"]:
        slope = grace_trend_cm_yr(geom)
        sev = None if slope is None else float(np.clip(0.5 - slope/4.0, 0, 1))  # -2 cm/yr -> 1.0
        out["aquifer_decline"] = _drv(sev, 0.30, DS["grace"], "regional",
            trend_cm_per_yr=None if slope is None else round(slope, 2),
            caveat="~300 km resolution - basin/regional signal")
    else:
        out["aquifer_decline"] = _drv(None, 0.30, "GRACE unavailable", "no-data")

    # RW - 1 - recharge potential
    ri, srcs = recharge_index(geom)
    out["recharge_worry"] = _drv(None if ri is None else 1 - ri, 0.20, " + ".join(srcs) or "unavailable",
                                  "computed" if ri is not None else "no-data",
                                  recharge_index=None if ri is None else round(ri, 3))

    # QD - excluded, disclosed
    out["quality_QD"] = _drv(None, 0.15, "no global groundwater-chemistry raster", "no-data",
                              caveat="excluded - weights renormalized; site lab data required")

    res = composite(out)
    res.update({"risk": "groundwater", "pfaf_id": pfaf, "runtime_s": round(time.time()-t0, 1)})
    return res

print("groundwater engine defined")

In [ ]:
# ✅ GATE 1b - PASS requires: Fresno (known Central-Valley depletion) shows high aquifer_decline;
#   weights renormalize with QD absent (attribution still sums ~100); every proxy carries its label.
for name, lat, lng in [("Fresno",36.7378,-119.7871), ("Chennai",13.0827,80.2707), ("Hamburg",53.5511,9.9937)]:
    g = groundwater_drivers(lat, lng)
    ssum = sum(g["attribution_pct"].values())
    print(f"\n=== {name} ===  runtime {g['runtime_s']}s   GH hazard: {g['hazard_0_5']} /5")
    for k, d in g["drivers"].items():
        print(f"    {k:<20} sev={d['severity']}  [{d['confidence']}]  {d.get('caveat','')}")
    print(f"  attribution sum: {ssum:.1f}%  | excluded: {g['excluded']}")
    assert abs(ssum - 100) < 1.5
    assert g["drivers"]["extraction_pressure"]["confidence"] == "proxy"
    assert "quality_QD" in g["excluded"]
fres = groundwater_drivers(36.7378, -119.7871)
ad = fres["drivers"]["aquifer_decline"]["severity"]
print(f"\nFresno aquifer_decline severity = {ad}  (expect elevated, >0.6, if GRACE loaded)")
print("GATE 1b: assertions passed")

## 1c - Flood engine
5 toolkit factors (weights: rainfall .25, runoff .25, catchment .20, soil saturation .15, urbanization .15):
- **Rainfall**: design storm = mean + 1 std of annual 1-day maxima (CHIRPS, 30 yr), normalized vs 120 mm/day reference
- **Runoff (SCS-CN)**: CN from HYSOGs250m x WorldCover via NRCS TR-55 lookup -> `S = 25400/CN - 254`,
  `Q = (P-0.2S)^2/(P+0.8S)`, severity = runoff coefficient `Q/P` for the design storm.
  If the HYSOGs probe failed: marked **pending asset** (no-data), weights renormalize - Gate 1c rule.
- **Catchment**: HydroSHEDS flow accumulation (log-scaled) + relative elevation above 1-km-buffer minimum
- **Soil saturation**: SMAP surface moisture, latest year vs record min-max
- **Urbanization**: GHSL built-up fraction 2020 + change since 2000

In [ ]:
# NRCS TR-55 representative curve numbers (AMC II) per WorldCover class x HSG A/B/C/D.
# Dual HSG classes (11-14 = A/D..D/D, high water table) conservatively treated as D.
CN_TABLE = {  # wc_class: (A, B, C, D)
    10:(36,60,73,79), 20:(35,56,70,77), 30:(49,69,79,84), 40:(70,80,87,90),
    50:(77,85,90,92), 60:(77,86,91,94), 70:(98,98,98,98), 80:(100,100,100,100),
    90:(85,85,85,85), 95:(85,85,85,85), 100:(63,77,85,88),
}

def _pt_val(img, lat, lng, scale, band=None):
    g = ee.Geometry.Point([lng, lat]).buffer(scale)
    d = img.reduceRegion(ee.Reducer.mean(), g, scale, maxPixels=1e9, bestEffort=True).getInfo()
    if not d: return None
    return d.get(band) if band else (list(d.values())[0] if d else None)

def _hsg_class(raw):
    """HYSOGs/HiHydroSoil raw value -> HSG 1-4. Handles HiHydroSoil's x10000 scaling
    and dual classes (11-14 -> D)."""
    if raw is None: return None
    v = float(raw)
    if v > 100: v = v / 10000.0          # HiHydroSoil v2.0 rasters are scaled by 10,000
    h = int(round(v))
    if h <= 0: return None
    return 4 if h > 4 else h             # dual classes 11-14 (and anything >4) -> D

def _wc_class(wc):
    """Snap a WorldCover buffer-mean to the nearest legal class key."""
    if wc is None: return None
    return min(CN_TABLE.keys(), key=lambda c: abs(c - wc))

def flood_drivers(lat, lng):
    """Each factor is individually try/except-wrapped: a server-side EE failure in one
    factor degrades THAT factor to no-data (with the error recorded) instead of
    crashing the whole engine - Gate 1c's honest-degradation rule."""
    t0 = time.time()
    geom, pfaf = basin_geom(lat, lng)
    if geom is None: return None
    site = ee.Geometry.Point([lng, lat])
    out = {}

    # 1. Rainfall - annual 1-day maxima series over the basin
    p_design = None
    try:
        if DS["chirps"]:
            mx = yearly(DS["chirps"], "precipitation", geom, 12, ee.Reducer.max(), 5000, CHIRPS_YEARS)
            vals = np.array([v for v in mx if v is not None], float)
            if len(vals) >= 10:
                p_design = float(vals.mean() + vals.std())
                sev = float(np.clip(p_design/120.0, 0, 1))   # 120 mm/day extreme-rain reference
                out["rainfall"] = _drv(sev, 0.25, DS["chirps"], "computed",
                                        design_storm_mm=round(p_design,1),
                                        method="mean+1std of annual 1-day maxima, /120mm ref")
    except Exception as e:
        out["rainfall"] = _drv(None, 0.25, DS["chirps"] or "CHIRPS", "no-data", error=str(e)[:160])
    if "rainfall" not in out:
        out["rainfall"] = _drv(None, 0.25, "CHIRPS unavailable", "no-data")

    # 2. Runoff SCS-CN (needs HYSOGs + WorldCover + design storm)
    try:
        if DS["hysogs"] and DS["worldcover"] and p_design:
            h = _hsg_class(_pt_val(ee.Image(DS["hysogs"]), lat, lng, 500))
            wc = _wc_class(_pt_val(ee.ImageCollection(DS["worldcover"]).first(), lat, lng, 500, "Map"))
            if h is not None and wc is not None:
                cn = CN_TABLE[wc][h-1]
                S = 25400.0/cn - 254.0
                P = p_design
                Q = ((P - 0.2*S)**2 / (P + 0.8*S)) if P > 0.2*S else 0.0
                out["runoff_SCS_CN"] = _drv(float(np.clip(Q/P, 0, 1)), 0.25,
                    f'{DS["hysogs"]} + {DS["worldcover"]}', "computed",
                    CN=cn, HSG=h, wc_class=wc, S_mm=round(S,1), Q_mm=round(Q,1),
                    method="NRCS TR-55, AMC II, dual HSG->D")
    except Exception as e:
        out["runoff_SCS_CN"] = _drv(None, 0.25, DS["hysogs"] or "HYSOGs", "no-data", error=str(e)[:160])
    if "runoff_SCS_CN" not in out:
        out["runoff_SCS_CN"] = _drv(None, 0.25,
            "HYSOGs250m pending asset (ORNL DAAC DOI 10.3334/ORNLDAAC/1566)", "no-data",
            caveat="pending asset - upload HYSOGs250m to GEE and re-run")

    # 3. Catchment - flow accumulation + relative elevation
    try:
        if DS["flowacc"] and DS["condem"]:
            acc = _pt_val(ee.Image(DS["flowacc"]), lat, lng, 500)
            zs  = _pt_val(ee.Image(DS["condem"]), lat, lng, 500)
            zmin_d = (ee.Image(DS["condem"]).reduceRegion(ee.Reducer.min(), site.buffer(1000), 500,
                      maxPixels=1e9, bestEffort=True).getInfo())
            zmin = list(zmin_d.values())[0] if zmin_d else None
            if acc is not None and zs is not None and zmin is not None:
                s_acc = float(np.clip(np.log10(acc + 1)/6.0, 0, 1))
                s_rel = float(np.clip(1 - (zs - zmin)/30.0, 0, 1))
                out["catchment"] = _drv(0.5*s_acc + 0.5*s_rel, 0.20,
                    f'{DS["flowacc"]} + {DS["condem"]}', "computed",
                    flow_acc_cells=int(acc), rel_elev_m=round(zs - zmin, 1))
    except Exception as e:
        out["catchment"] = _drv(None, 0.20, "HydroSHEDS", "no-data", error=str(e)[:160])
    if "catchment" not in out:
        out["catchment"] = _drv(None, 0.20, "HydroSHEDS unavailable", "no-data")

    # 4. Soil saturation - SMAP latest year vs record. 3-month window (Oct-Dec) keeps the
    # request light: SPL4 is 3-hourly, so a 12-mo x 10-yr composite can time out server-side.
    try:
        if DS["smap"]:
            sm = yearly(DS["smap"], "sm_surface", geom, 3, ee.Reducer.mean(), 10000,
                        list(range(2016, 2026)))
            vals = [v for v in sm if v is not None]
            if len(vals) >= 5 and sm[-1] is not None:
                lo, hi = min(vals), max(vals)
                sev = float((sm[-1]-lo)/(hi-lo)) if hi > lo else 0.5
                out["soil_saturation"] = _drv(sev, 0.15, DS["smap"], "computed",
                                               caveat="SMAP top ~5cm; Oct-Dec window each year")
    except Exception as e:
        out["soil_saturation"] = _drv(None, 0.15, DS["smap"] or "SMAP", "no-data", error=str(e)[:160])
    if "soil_saturation" not in out:
        out["soil_saturation"] = _drv(None, 0.15, "SMAP unavailable", "no-data")

    # 5. Urbanization - GHSL built fraction + change. Epoch size is checked client-side
    # BEFORE mosaic: reduceRegion on a band-less (empty-mosaic) image errors server-side.
    try:
        if DS["ghsl"]:
            ghsl = ee.ImageCollection(DS["ghsl"]).select("built_surface")
            def _epoch(y0, y1):
                sub = ghsl.filterDate(f"{y0}-01-01", f"{y1}-01-01")
                if sub.size().getInfo() == 0: return None
                return _pt_val(sub.mosaic(), lat, lng, 1000, "built_surface")
            b20 = _epoch(2019, 2021)
            b00 = _epoch(1999, 2001)
            if b20 is not None:
                f20 = float(np.clip(b20/10000.0, 0, 1))     # m2 per 100m cell -> fraction
                delta = float(np.clip((b20-(b00 or 0))/10000.0, 0, 1))
                out["urbanization"] = _drv(float(np.clip(0.7*f20 + 0.3*delta, 0, 1)), 0.15,
                                            DS["ghsl"], "computed",
                                            built_frac_2020=round(f20,3), built_change=round(delta,3))
    except Exception as e:
        out["urbanization"] = _drv(None, 0.15, DS["ghsl"] or "GHSL", "no-data", error=str(e)[:160])
    if "urbanization" not in out:
        out["urbanization"] = _drv(None, 0.15, "GHSL unavailable", "no-data")

    res = composite(out)
    res.update({"risk": "flood", "pfaf_id": pfaf, "runtime_s": round(time.time()-t0, 1)})
    return res

print("flood engine defined (per-factor error isolation)")

In [ ]:
# ✅ GATE 1c - PASS requires: Chennai (2015-flood city) scores high; CN sane for its land cover
#   (urban -> ~90s, forest -> ~60s); if hysogs MISSING, runoff shows 'pending asset' and the
#   other 4 factors still compute with renormalized weights (honest, not silently skipped).
for name, lat, lng in [("Chennai",13.0827,80.2707), ("Hamburg",53.5511,9.9937), ("Riyadh",24.7136,46.6753)]:
    f = flood_drivers(lat, lng)
    aq = aq_scores(lat, lng)
    ssum = sum(f["attribution_pct"].values())
    print(f"\n=== {name} ===  runtime {f['runtime_s']}s")
    print(f"  modelled flood hazard: {f['hazard_0_5']} /5  | Aqueduct rfr: {aq.get('rfr_score')}  cfr: {aq.get('cfr_score')}")
    for k, d in f["drivers"].items():
        extra = {kk: vv for kk, vv in d.items() if kk not in ('severity','weight','source','confidence')}
        print(f"    {k:<18} sev={d['severity']}  [{d['confidence']}]  {extra}")
    print(f"  attribution sum: {ssum:.1f}%  excluded: {f['excluded']}")
    assert abs(ssum - 100) < 1.5
cn_chennai = flood_drivers(13.0827, 80.2707)["drivers"]["runoff_SCS_CN"]
print("\nChennai CN sanity:", cn_chennai.get("CN", "pending asset"), "(urban expect ~85-92)")
print("GATE 1c: assertions passed")

## Export - precompute all drivers for the 5 demo sites
Produces `drivers_demo.json` -> seeds the Supabase `driver_cache` table in Phase 2
(and doubles as the offline demo fallback).

In [ ]:
all_drivers = {}
for s in SITES:
    print(f"computing {s['name']} ...")
    all_drivers[s["id"]] = {
        "site": s,
        "drought":     drought_drivers(s["lat"], s["lng"]),
        "groundwater": groundwater_drivers(s["lat"], s["lng"]),
        "flood":       flood_drivers(s["lat"], s["lng"]),
    }

assert "-9999" not in json.dumps(all_drivers), "-9999 leak!"
with open("drivers_demo.json", "w") as fp:
    json.dump(all_drivers, fp, indent=1)
print("\nwrote drivers_demo.json")
try:
    from google.colab import files; files.download("drivers_demo.json")
except Exception: pass

---
**Phase-1 outputs:** three driver engines with per-driver `source` + `confidence`, renormalized
weights over available data, attribution % (labeled *estimated contribution to the modelled score*),
and `drivers_demo.json` for Supabase seeding.

**Data:** WRI Aqueduct 4.0, CHIRPS (UCSB CHC), GLDAS & GRACE & SMAP & MOD13/MOD16 (NASA),
HydroSHEDS (WWF), ESA WorldCover, GHSL (JRC), SoilGrids (ISRIC), HYSOGs250m (ORNL DAAC) - all via
Google Earth Engine, free with attribution.

**Next:** Gates 1a/1b/1c pass -> Phase 2 (Supabase schema + FastAPI GEE-compute service).